## Inventory Optimization & Scenario Simulation

#### Notebook Objective

Convert demand forecasts into actionable inventory decisions by:

- Designing reorder rules

- Calculating safety stock

- Stress-testing inventory using demand scenarios

- Quantifying financial impact of stockouts vs holding costs

#### Key Assumptions
For this analysis, we assume:

- Although transactions occur throughout the day, we approximate daily average demand using historical sales. Inventory review is assumed to occur once per day at close of business.

- Lead time is constant

- Demand variability is captured using historical data

- Forecast errors are unavoidable → safety stock is required

These assumptions reflect realistic retail operations.

#### Importing Libraries and Loading Dataset

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df=pd.read_csv("cleaned_BrightMart_retail_dataset.csv")
df.head()

,Category,Item,Price Per Unit,Quantity,Payment Method,Location,Transaction Date,Discount Applied,Season,Month,Weekday,Stock On Hand,Promotion,Revenue
0,Patisserie,Item_10_Pat,18.5,10.0,Digital Wallet,Online,2024-04-08,True,Spring,April,Monday,152,10,166.50
1,Milk Products,Item_17_Milk,29.0,9.0,Digital Wallet,Online,2023-07-23,True,Summer,July,Sunday,142,10,234.90
2,Butchers,Item_12_But,21.5,2.0,Credit Card,Online,2022-10-05,True,Fall,October,Wednesday,64,10,38.70
3,Beverages,Item_16_Bev,27.5,9.0,Credit Card,Online,2022-05-07,True,Spring,May,Saturday,156,10,222.75
4,Food,Item_6_Food,12.5,7.0,Digital Wallet,Online,2022-10-02,False,Fall,October,Sunday,121,0,87.50


#### Step 1 : Prepare Demand Input for Inventory Decisions


- Inventory decisions are based on units sold, not revenue

- Average daily demand is calculated per category

#### Average Daily Demand

What it means:
- Average number of units sold per day for each category.

Why it matters:
- Inventory is consumed daily. Planning starts with how fast stock moves.

In [ ]:
# Daily average demand per category
daily_demand= df.groupby('Category')['Quantity'].mean().reset_index()
daily_demand.columns = ['Category', 'Avg_Daily_Demand']
daily_demand

,Category,Avg_Daily_Demand
0,Beverages,5.605616
1,Butchers,5.463010
2,Computers And Electric Accessories,5.621309
3,Electric Household Essentials,5.505343
4,Food,5.587531
5,Furniture,5.567568
6,Milk Products,5.533460
7,Patisserie,5.539921


#### Step 2: Defining Business Parameters

In [ ]:
Lead_Time_Days= 7 #supplier lead time
Service_Level= 0.95 #desired service level(95%)
Holding_Cost_Rate= 0.20 #Annual holding cost of inventory(%of costing value)

- Lead time = supplier delivery delay

- Service level = stockout tolerance

- Holding cost = capital + storage + risk

#### Step 3 — Calculating Demand Variability

- Forecasts are never perfect.
- Safety stock protects against demand uncertainty.

#### Demand Standard Deviation (σ)

What it means:
- How much daily demand fluctuates around the average.

Low σ → stable demand

High σ → unpredictable demand

This captures uncertainty, not volume.

In [ ]:
demand_std= df.groupby("Category")['Quantity'].std().reset_index()
demand_std.columns = ['Category', 'Demand_STD']
demand_std

,Category,Demand_STD
0,Beverages,2.811474
1,Butchers,2.828636
2,Computers And Electric Accessories,2.781323
3,Electric Household Essentials,2.787405
4,Food,2.728278
5,Furniture,2.823140
6,Milk Products,2.800847
7,Patisserie,2.761837


#### Step 4 — Safety Stock Calculation

- Z (1.65) → how safe the business wants to be (95% service level)

- σ (Demand_STD) → how unpredictable demand is

- √Lead Time → longer supplier wait increases uncertainty

What safety stock does:
- It protects the business from unexpected demand spikes or forecast errors during supplier lead time.

#### Formula:
Z * Standard_Deviation * root_square of (lead_time)

In [ ]:
Z=1.65
inventory_policy=daily_demand.merge(demand_std, on='Category')
inventory_policy['Safety_Stock']= (Z*inventory_policy['Demand_STD']*np.sqrt(Lead_Time_Days)).round()
inventory_policy

,Category,Avg_Daily_Demand,Demand_STD,Safety_Stock
0,Beverages,5.605616,2.811474,12.0
1,Butchers,5.463010,2.828636,12.0
2,Computers And Electric Accessories,5.621309,2.781323,12.0
3,Electric Household Essentials,5.505343,2.787405,12.0
4,Food,5.587531,2.728278,12.0
5,Furniture,5.567568,2.823140,12.0
6,Milk Products,5.533460,2.800847,12.0
7,Patisserie,5.539921,2.761837,12.0


#### 1. Safety Stock Is Identical Across Categories

- All categories require 12 units of safety stock
- Safety stock values are similar across categories because demand variability (standard deviation), service level, and lead time are consistent. In real operations, category-specific lead times would produce differentiated buffers

This happens because:

- Demand variability (σ = 2.7–2.8) is similar

- Same service level and lead time are applied

Insight:
- At a high level, BrightMart’s categories show similar demand volatility, enabling standardized buffer rules.

#### Reorder Point (ROP) Calculation

#### ROP defines when to place a new order, not how much.

It ensures that:

- Expected demand during supplier wait is covered

- Plus extra buffer (safety stock)

#### ROP=(Avg Daily Demand×Lead Time)+Safety Stock

In [ ]:
inventory_policy['Reorder_Point']= (inventory_policy['Avg_Daily_Demand']*Lead_Time_Days + inventory_policy['Safety_Stock']).round()
inventory_policy

,Category,Avg_Daily_Demand,Demand_STD,Safety_Stock,Reorder_Point
0,Beverages,5.605616,2.811474,12.0,51.0
1,Butchers,5.463010,2.828636,12.0,50.0
2,Computers And Electric Accessories,5.621309,2.781323,12.0,51.0
3,Electric Household Essentials,5.505343,2.787405,12.0,51.0
4,Food,5.587531,2.728278,12.0,51.0
5,Furniture,5.567568,2.823140,12.0,51.0
6,Milk Products,5.533460,2.800847,12.0,51.0
7,Patisserie,5.539921,2.761837,12.0,51.0



#### Reorder Point Clusters Around 50–51 Units

Most categories have ROP = 50–51 units

Indicates:

- Similar daily demand (5.5 units/day)

- Similar replenishment risk profile

Insight:
- Inventory policies can be simplified operationally while still being data-driven.
- When stock falls below ROP → place order

- Prevents stockouts during lead time

#### Step 6 — Demand Scenario Simulation
Demand is not static. Promotions and events cause spikes.

To test inventory robustness, we simulate demand shocks:

- Normal Demand → average conditions

- Promotional Demand (+30%) → discount campaigns

- Surge Demand (+50%) → festivals / viral demand

This is stress testing, not forecasting.

Promotional_Demand × Lead_Time > Reorder_Point

- If TRUE → stock will run out before replenishment arrives.

In [ ]:
inventory_policy['Normal_Demand']= inventory_policy['Avg_Daily_Demand']
inventory_policy['Promotional_Demand']= inventory_policy['Avg_Daily_Demand']*1.3
inventory_policy['Surge_Demand']= inventory_policy['Avg_Daily_Demand']*1.5
inventory_policy

,Category,Avg_Daily_Demand,Demand_STD,Safety_Stock,Reorder_Point,Normal_Demand,Promotional_Demand,Surge_Demand
0,Beverages,5.605616,2.811474,12.0,51.0,5.605616,7.287301,8.408424
1,Butchers,5.463010,2.828636,12.0,50.0,5.463010,7.101913,8.194515
2,Computers And Electric Accessories,5.621309,2.781323,12.0,51.0,5.621309,7.307702,8.431964
3,Electric Household Essentials,5.505343,2.787405,12.0,51.0,5.505343,7.156945,8.258014
4,Food,5.587531,2.728278,12.0,51.0,5.587531,7.263791,8.381297
5,Furniture,5.567568,2.823140,12.0,51.0,5.567568,7.237838,8.351351
6,Milk Products,5.533460,2.800847,12.0,51.0,5.533460,7.193497,8.300189
7,Patisserie,5.539921,2.761837,12.0,51.0,5.539921,7.201898,8.309882


#### 3. Promotions Create Stockout Risk for Select Categories

Stockout risk = TRUE for:

- Beverages

- Computers & Electric Accessories

Despite similar averages, these categories break under promotion demand.

Insight:
- Even small demand differences matter when promotions are applied.

#### Step 7 — Stockout Risk Assessment

Stockout Cost=Lost Units×Price

Represents lost revenue due to unmet demand.

In [ ]:
inventory_policy['Stockout_Risk']= (inventory_policy['Promotional_Demand']*Lead_Time_Days> inventory_policy['Reorder_Point'])
inventory_policy[['Category','Stockout_Risk']]

,Category,Stockout_Risk
0,Beverages,True
1,Butchers,False
2,Computers And Electric Accessories,True
3,Electric Household Essentials,False
4,Food,False
5,Furniture,False
6,Milk Products,False
7,Patisserie,False


#### 4. Stockout Cost Is 3× Higher Than Holding Cost

Example (Beverages):

- Stockout Cost ≈ $170

- Holding Cost ≈ $56

This pattern repeats across categories.

Insight:
- From a cost perspective, slight overstocking is cheaper than losing sales.

#### Step 8 — Cost Impact Simulation
Assumptions

- Lost sale cost = average unit price

- Overstock cost = holding cost

#### Holding Cost=Safety Stock×Price×Holding Rate

Represents cost of storing extra inventory.

In [ ]:
avg_price= df.groupby('Category')['Price Per Unit'].mean().reset_index()
inventory_policy= inventory_policy.merge(avg_price, on="Category")
inventory_policy['Lost_Units'] = (inventory_policy['Promotional_Demand'] * Lead_Time_Days - inventory_policy['Reorder_Point']).clip(lower=0)
inventory_policy['Stockout_Cost'] = (inventory_policy['Lost_Units'] * inventory_policy['Price Per Unit']).round()
inventory_policy['Holding_Cost']= (inventory_policy['Safety_Stock']*inventory_policy['Price Per Unit']* Holding_Cost_Rate).round()

#### 5. High-Risk Categories Are Not the Highest Demand Ones

Risk depends on:

- Demand variability

- Lead time

- Promotion intensity

- Not just average sales.

Insight:
- Inventory risk must be evaluated holistically, not by volume alone.
- Losing a sale usually costs MORE than holding extra stock

In [ ]:
inventory_policy

,Category,Avg_Daily_Demand,Demand_STD,Safety_Stock,Reorder_Point,Normal_Demand,Promotional_Demand,Surge_Demand,Stockout_Risk,Price Per Unit,Lost_Units,Stockout_Cost,Holding_Cost
0,Beverages,5.605616,2.811474,12.0,51.0,5.605616,7.287301,8.408424,True,23.314419,0.011104,0.0,56.0
1,Butchers,5.463010,2.828636,12.0,50.0,5.463010,7.101913,8.194515,False,25.114869,0.000000,0.0,60.0
2,Computers And Electric Accessories,5.621309,2.781323,12.0,51.0,5.621309,7.307702,8.431964,True,23.154263,0.153915,4.0,56.0
3,Electric Household Essentials,5.505343,2.787405,12.0,51.0,5.505343,7.156945,8.258014,False,24.385913,0.000000,0.0,59.0
4,Food,5.587531,2.728278,12.0,51.0,5.587531,7.263791,8.381297,False,23.093563,0.000000,0.0,55.0
5,Furniture,5.567568,2.823140,12.0,51.0,5.567568,7.237838,8.351351,False,23.363696,0.000000,0.0,56.0
6,Milk Products,5.533460,2.800847,12.0,51.0,5.533460,7.193497,8.300189,False,21.417781,0.000000,0.0,51.0
7,Patisserie,5.539921,2.761837,12.0,51.0,5.539921,7.201898,8.309882,False,23.066964,0.000000,0.0,55.0


In [ ]:
inventory_policy.to_csv("inventory_policy.csv", index=False)

#### Key Insights
- Stockouts cost significantly more than holding inventory

- Promotions increase demand risk even for average-performing categories

- Inventory readiness determines realized sales more than discounts

- Standardized inventory rules are feasible due to similar demand variability

#### Final Business Recommendations
#### Inventory Policy
- Maintain safety stock of ~12 units

- Trigger replenishment at ~50 units

#### Promotion Planning

Increase buffers for:

- Beverages

- Electric Accessories

- Do not launch promotions without inventory readiness

#### Cost Strategy

- Accept moderate holding cost

- Prioritize service continuity over lean inventory

#### Inventory availability, not discounts alone, determines realized sales.
By aligning demand forecasts with reorder rules and safety stock, BrightMart can reduce lost revenue, stabilize operations, and make promotion decisions with quantified risk.